In [1]:
import sys
import csv
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from scipy.stats import norm


# Project-specific paths — update these to match your local setup
XS_LIB_PATH   = '/home/paule/open_mc_projects/xs_lib/endfb-vii.1-hdf5/neutron'
SRC_PATH       = '/home/paule/open_mc_projects/MC-1D_DT/structured_code'
VECTFIT_DIR    = Path(SRC_PATH) / 'src/vectfit_data'
OPENMC_FILE    = Path(SRC_PATH) / 'validation/method_validation/openmc_results_point.csv'

sys.path.extend([XS_LIB_PATH, SRC_PATH])

import geometry_constructor
from src.geometry_classes import Geometry, Source, Material
import src.parallel as parallel
import src.export_print_csv as xp_sim
import src.illustration

['/home/paule/anaconda3/envs/vectfit39/lib/python39.zip', '/home/paule/anaconda3/envs/vectfit39/lib/python3.9', '/home/paule/anaconda3/envs/vectfit39/lib/python3.9/lib-dynload', '', '/home/paule/anaconda3/envs/vectfit39/lib/python3.9/site-packages', '/home/paule/open_mc_projects/xs_lib/endfb-vii.1-hdf5/neutron', '/home/paule/open_mc_projects/MC-1D_DT/structured_code', '/home/paule/open_mc_projects/windowed_multipole/02_working_notebook_vectfit']


In [ ]:
xs_methods = ['serpent', 'discrete', 'vectfit', 'sqrtT_E']
maj_mat_methods = ['maj_mat', 'simple']
access_methods = ['fly', 'reconr']

N_NEUTRONS  = 20000  # neutrons per batch
N_BATCHES   = 100   # number of independent batches 

def _is_forbidden(xs_method, maj_mat_method, access_method) -> bool:
    if access_method == "fly" and maj_mat_method == "simple":
        return True
    if access_method == "fly" and xs_method in ("discrete", "serpent"):
        return True
    return False
 
 
REQUIRED_FILES = ["cross_batch_statistics_corrected.csv", "memory_summary.txt"]
ERROR_FILES    = ["error_log.txt"]
 
results = {}
 
for xs_method in xs_methods:
    for maj_mat_method in maj_mat_methods:
        for access_method in access_methods:
 
            if _is_forbidden(xs_method, maj_mat_method, access_method):
                print(f"[skip] xs={xs_method}, maj={maj_mat_method}, access={access_method}")
                continue
 
            key        = (xs_method, maj_mat_method, access_method)
            output_dir = f"data/{xs_method}_{maj_mat_method}_{access_method}_{N_NEUTRONS}x{N_BATCHES}"
            base       = Path(output_dir)
 
            # ── check whether we need to run ──────────────────────────────
            if not base.exists():
                base.mkdir(parents=True, exist_ok=True)
                to_run = True
            else:
                missing = [f for f in REQUIRED_FILES if not (base / f).exists()]
                errored = [f for f in ERROR_FILES    if     (base / f).exists()]
                if errored:
                    print(f"[skip] error log found in {output_dir}")
                    to_run = False
                elif missing:
                    print(f"[rerun] incomplete output in {output_dir}, missing: {missing}")
                    to_run = True
                else:
                    print(f"[skip] complete output already exists in {output_dir}")
                    to_run = False
 
            if not to_run:
                continue
 
            # ── run ───────────────────────────────────────────────────────
            print(f"\n{'='*60}")
            print(f"[run]  xs={xs_method}, maj={maj_mat_method}, access={access_method}")
            print(f"{'='*60}")
 
            geometry    = None
            batch_stats = None
 
            try:
                geometry, source = geometry_constructor.create_geometry_PWR_fresh_fuel(
                    maj_mat_method = maj_mat_method,
                    maj_xs_method  = xs_method,
                    access_method  = access_method,
                    mode           = "analysis",
                    neutron_nbr    = N_NEUTRONS,
                )
 
                batch_stats = parallel.run_batch(
                    geometry,
                    source,
                    N_BATCHES,
                    mode      = "parallel",
                    n_workers = 20,
                )
 
            except Exception as e:
                print(f"[error] {key}: {e}")
                with open(base / "error_log.txt", "w") as f:
                    f.write(f"xs={xs_method}, maj={maj_mat_method}, access={access_method}\n")
                    f.write(f"{e}\n")
                continue
 
            # ── export ────────────────────────────────────────────────────
            else:
                results[key] = batch_stats
 
                try:
                    xp_sim.export_cross_batch_stats(
                        batch_stats     = batch_stats,
                        geom            = geometry,
                        print_to_console = False,
                        save_csv        = True,
                        output_dir      = output_dir,
                    )
                except Exception as e:
                    print(f"[error] export failed for {key}: {e}")

                try:
                    xp_sim.export_memory_stats(geom=geometry, output_dir=output_dir)
                except Exception as e:
                    print(f"Memory export failed: {e}")



[skip] xs=serpent, maj=maj_mat, access=fly

[run]  xs=serpent, maj=maj_mat, access=reconr
[Material] Processing nuclide pair: H1 (Density: 4.75e+22)
[Material] Processing nuclide pair: O16 (Density: 2.37e+22)

 [Memory] Tracker started (poll interval: 1 ms)

 [Memory] Tracker stopped.
[Material] Processing nuclide pair: H1 (Density: 4.75e+22)
[Material] Processing nuclide pair: O16 (Density: 2.37e+22)

 [Memory] Tracker started (poll interval: 1 ms)

 [Memory] Tracker stopped.
[Material] Processing nuclide pair: H1 (Density: 4.75e+22)
[Material] Processing nuclide pair: O16 (Density: 2.37e+22)

 [Memory] Tracker started (poll interval: 1 ms)

 [Memory] Tracker stopped.
[Material] Processing nuclide pair: U235 (Density: 2.66e+20)
[Material] Processing nuclide pair: U238 (Density: 2.60e+22)
[Material] Processing nuclide pair: O16 (Density: 4.64e+22)

 [Memory] Tracker started (poll interval: 1 ms)

 [Memory] Tracker stopped.
[Material] Processing nuclide pair: U235 (Density: 2.66e+20)
[M